# Recurrent Event Network (RENet) on Temporal Knowledge Graphs

Link Prediction on ICEWS18: Predicting future dynamic links using recurrent graph convolutional networks. This notebook implements the approach with `RENet`, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `RENet` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import models as k3_models

title = "Recurrent Event Network (RENet) on Temporal Knowledge Graphs"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. RENet Model
num_nodes = 100
num_rels = 20
seq_len = 5
batch_size = 3
k3_model = k3_models.RENet(
    num_nodes=num_nodes,
    num_rels=num_rels,
    hidden_channels=32,
    seq_len=seq_len,
)

# 2. Test forward pass
# RENet scores (sub, rel, obj) triples using each entity's aggregated
# history over the last `seq_len` timesteps, so a forward pass needs that
# history alongside the triples themselves: `h_*` are the historical entity
# ids, `h_*_t` their timestep within the window, and `h_*_batch` which
# triple in the batch they belong to.
rng = np.random.default_rng(0)
dummy_sub = ops.convert_to_tensor([0, 1, 2], dtype="int64")
dummy_rel = ops.convert_to_tensor([0, 1, 2], dtype="int64")
dummy_obj = ops.convert_to_tensor([3, 4, 5], dtype="int64")

history_len = 12
h_sub = ops.convert_to_tensor(rng.integers(0, num_nodes, size=history_len), dtype="int64")
h_sub_t = ops.convert_to_tensor(rng.integers(0, seq_len, size=history_len), dtype="int64")
h_sub_batch = ops.convert_to_tensor(rng.integers(0, batch_size, size=history_len), dtype="int64")
h_obj = ops.convert_to_tensor(rng.integers(0, num_nodes, size=history_len), dtype="int64")
h_obj_t = ops.convert_to_tensor(rng.integers(0, seq_len, size=history_len), dtype="int64")
h_obj_batch = ops.convert_to_tensor(rng.integers(0, batch_size, size=history_len), dtype="int64")

log_prob_obj, log_prob_sub = k3_model(
    dummy_sub, dummy_rel, dummy_obj,
    h_sub, h_sub_t, h_sub_batch,
    h_obj, h_obj_t, h_obj_batch,
)
print(f"RENet object-prediction log-probs shape: {log_prob_obj.shape} (Expected: (3, {num_nodes}))")
print(f"RENet subject-prediction log-probs shape: {log_prob_sub.shape} (Expected: (3, {num_nodes}))")

print("\n✓ K3-Node RENet execution completed successfully!")